# DATA 715 | Apache Spark: RDDs, DataFrames, and MapReduce
**Hands-on Google Colab lab**  

**Learning objectives:** Explain distributed partitions, immutability, lineage and fault tolerance; distinguish transformations from actions; implement `map`, `filter`, `flatMap`, `reduce` and `reduceByKey`; compare RDD and DataFrame APIs; interpret the role of typed Datasets in Scala and Java.

Run the notebook **top to bottom**. Cells marked *intentional error* catch the exception to let the notebook continue. The examples operate on small instructional datasets; avoid `collect()` on large distributed data.

## Part 1. Set up Spark in Google Colab
Spark 4.x requires a compatible Java runtime. The installation below targets Colab (Linux), installs Java 17, and initializes only **one** Spark session. Run setup once per runtime. If package installation changes an already imported PySpark version, restart the runtime and rerun.

In [ ]:
# Cell 1: Install runtime dependencies (Google Colab)
import sys
if 'google.colab' in sys.modules:
    !apt-get update -qq
    !apt-get install -y openjdk-17-jdk-headless -qq > /dev/null
    %pip install -q "pyspark>=4.0,<5.0"
else:
    print("Non-Colab environment: ensure Java 17+ and PySpark 4.x are installed.")

In [ ]:
# Cell 2: Configure and start Spark once
import os, sys, subprocess
from pyspark.sql import SparkSession, functions as F
if 'google.colab' in sys.modules:
    java_path = subprocess.check_output(['readlink', '-f', '/usr/bin/java'], text=True).strip()
    os.environ['JAVA_HOME'] = os.path.dirname(os.path.dirname(java_path))
os.environ['PYSPARK_PYTHON'] = sys.executable
spark = (SparkSession.builder.appName('DATA715 Spark Lab')
         .master('local[*]').config('spark.ui.showConsoleProgress', 'false')
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel('ERROR')
print('Spark:', spark.version, '| Python:', sys.version.split()[0], '| Java home:', os.environ.get('JAVA_HOME', 'runtime default'))
print('Default parallelism:', sc.defaultParallelism)

## Part 2. Understanding RDDs
**RDD** means *Resilient Distributed Dataset*. An RDD is a partitioned collection distributed across executors. It is **immutable**: `map()` and `filter()` return new RDDs rather than changing the original. Transformations are generally **lazy**, and actions such as `collect()`, `count()` and `take()` trigger execution.

**Teaching distinction:** Immutability does not itself mean fault tolerance. Spark tracks **lineage**, which it can use to reconstruct lost partitions when upstream inputs remain available.

In [ ]:
# Cell 3: Create an RDD and inspect its partitions
rdd_numbers = sc.parallelize([1, 2, 3, 4, 5, 6], 2)
print('Values:', rdd_numbers.collect())
print('Number of partitions:', rdd_numbers.getNumPartitions())
print('Partition contents:', rdd_numbers.glom().collect())

In [ ]:
# Cell 4: Immutability using a safe, intentional error
python_list = [1, 2, 3, 4, 5]
python_list[0] = 100
print('Mutable Python list:', python_list)

try:
    rdd_numbers[0] = 100  # RDDs do not support item assignment
except TypeError as exc:
    print('Expected RDD error:', type(exc).__name__, '-', exc)
print('Original RDD unchanged:', rdd_numbers.collect())

In [ ]:
# Cell 5: A transformation creates a new RDD
# map is lazy; collect is an action.
doubled_rdd = rdd_numbers.map(lambda x: x * 2)
squared_rdd = doubled_rdd.map(lambda x: x ** 2)
print('Original:', rdd_numbers.collect())
print('Doubled: ', doubled_rdd.collect())
print('Squared: ', squared_rdd.collect())
assert rdd_numbers.collect() == [1, 2, 3, 4, 5, 6]

### Resiliency and lineage
The next cell displays the dependency chain. Spark may recompute a lost partition from its parent partitions if the original data source is still accessible. **This is a lineage illustration, not a simulated executor failure.** `unpersist()` discards cached blocks and is safe for illustrating recomputation, but does not simulate a worker crash.

In [ ]:
# Cell 6: Inspect lineage and demonstrate recomputation after unpersist
print('RDD lineage:')
lineage = squared_rdd.toDebugString()
print(lineage.decode('utf-8') if isinstance(lineage, bytes) else lineage)
squared_rdd.persist()
print('First action (populate cache):', squared_rdd.collect())
squared_rdd.unpersist(blocking=True)
print('After unpersist (recomputed):', squared_rdd.collect())
print('Same result:', squared_rdd.collect() == [4, 16, 36, 64, 100, 144])

## Part 3. RDD transformations and actions
A **transformation** returns an RDD; an **action** starts computation or returns a result to the driver. For an introductory class, compare `map` (one output per input), `flatMap` (zero or more outputs), and `filter` (retain selected elements).

In [ ]:
# Cell 7: map(), filter(), flatMap(), reduce()
print('map, double:', rdd_numbers.map(lambda x: x * 2).collect())
print('filter, even:', rdd_numbers.filter(lambda x: x % 2 == 0).collect())
sentences = sc.parallelize(['spark is fast', 'spark is scalable', 'rdd is powerful'])
print('flatMap, words:', sentences.flatMap(lambda line: line.split()).collect())
print('reduce, sum:', rdd_numbers.reduce(lambda a, b: a + b))

## Part 4. MapReduce in Spark
A classic word count workflow first produces `(word, 1)` pairs, then aggregates by word using `reduceByKey`. Spark can combine values locally before a shuffle; results are not inherently sorted.

In [ ]:
# Cell 8: Word count, step by step
text_data = ['cat dog cat bird', 'dog cat fish', 'bird fish cat']
rdd_text = sc.parallelize(text_data, 2)
words = rdd_text.flatMap(lambda line: line.split())
pairs = words.map(lambda word: (word, 1))
word_counts = pairs.reduceByKey(lambda a, b: a + b)
print('Words:', words.collect())
print('Pairs:', pairs.collect())
print('Counts, sorted:', sorted(word_counts.collect()))
print('Counts by frequency:', word_counts.sortBy(lambda x: (-x[1], x[0])).collect())

In [ ]:
# Cell 9: Sum and average student scores
scores = [('Alice', 85), ('Bob', 90), ('Alice', 95), ('Bob', 88),
          ('Charlie', 91), ('Alice', 78)]
rdd_scores = sc.parallelize(scores)
totals = rdd_scores.reduceByKey(lambda a, b: a + b)
sum_and_count = (rdd_scores.mapValues(lambda score: (score, 1))
                 .reduceByKey(lambda a, b: (a[0] + b[0], a[1] + b[1])))
averages = sum_and_count.mapValues(lambda x: round(x[0] / x[1], 2))
print('Totals:', sorted(totals.collect()))
print('Average:', sorted(averages.collect()))

In [ ]:
# Cell 10: Sales grouped by category
sales_data = [('Books', 20), ('Books', 35), ('Games', 60),
              ('Books', 15), ('Games', 25), ('Music', 40)]
sales_rdd = sc.parallelize(sales_data)
sales_totals = sales_rdd.reduceByKey(lambda a, b: a + b)
print('Totals:', sorted(sales_totals.collect()))

## Part 5. RDD versus DataFrame versus Dataset

| Feature | RDD | DataFrame | Typed Dataset |
|:--|:--|:--|:--|
| Data model | Distributed objects | Named, schema-defined columns | Schema plus typed records |
| Immutable transformations | Yes | Yes | Yes |
| Catalyst query optimization | No | Yes | Yes |
| Column names and SQL | Not intrinsic | Yes | Yes |
| Python API | Yes | Yes | **Not available for typed Dataset** |
| Scala/Java API | Yes | Yes | Yes |

**Clarification:** PySpark DataFrames use Spark's Dataset-of-Rows engine, but **PySpark does not expose the strongly typed Scala/Java `Dataset[T]` API**. Both RDDs and DataFrames are immutable. RDDs may contain structured tuples but do not enforce a Spark SQL schema.

In [ ]:
# Cell 11: Same input represented as RDD and DataFrame
people = [(1, 'Alice', 25), (2, 'Bob', 30), (3, 'Charlie', 35)]
people_rdd = sc.parallelize(people)
people_df = spark.createDataFrame(people, ['ID', 'Name', 'Age'])
print('RDD (tuples):', people_rdd.collect())
print('DataFrame schema:')
people_df.printSchema()
people_df.show()

In [ ]:
# Cell 12: Same problem, two APIs
rdd_adults = (people_rdd.filter(lambda row: row[2] >= 30)
              .map(lambda row: (row[1], row[2])))
df_adults = people_df.filter(F.col('Age') >= 30).select('Name', 'Age')
print('RDD result:', rdd_adults.collect())
print('DataFrame result:')
df_adults.show()
print('DataFrame physical plan:')
df_adults.explain(mode='simple')

In [ ]:
# Cell 13: DataFrame immutability and SQL
older_df = people_df.withColumn('Age', F.col('Age') + 1)
print('Original ages:', [r.Age for r in people_df.orderBy('ID').collect()])
print('New ages:     ', [r.Age for r in older_df.orderBy('ID').collect()])
people_df.createOrReplaceTempView('people')
spark.sql('SELECT Name, Age FROM people WHERE Age >= 30 ORDER BY Name').show()

### Typed Dataset example: Scala (illustration, not a Python cell)
The following is for a Scala Spark environment. A `Dataset[Person]` offers compile-time checks for typed record properties.

```scala
import spark.implicits._
case class Person(ID: Int, Name: String, Age: Int)
val ds = Seq(Person(1, "Alice", 25), Person(2, "Bob", 30)).toDS()
ds.filter(person => person.Age >= 30).show()
// ds.filter(person => person.Height > 170) // compile-time error
```

## Part 6. Guided practice
Try the exercises before revealing or running the solution cells. The exercises deliberately reuse the operators introduced above.

### Exercise 1. Transform numbers
Create an RDD containing 1 through 10. Square each number, retain results greater than 20, and display them in ascending order.

In [ ]:
# Exercise 1: Write your solution here
# YOUR CODE HERE

**Exercise 1 solution**

In [ ]:
practice_rdd = sc.parallelize(range(1, 11))
result = practice_rdd.map(lambda x: x ** 2).filter(lambda x: x > 20)
print(sorted(result.collect()))
assert sorted(result.collect()) == [25, 36, 49, 64, 81, 100]

### Exercise 2. Word count
Count words in these lines: `data science is fun`, `big data is big`, `science uses data`. Use `flatMap`, `map`, and `reduceByKey`.

In [ ]:
# Exercise 2: Write your solution here
practice_text = ['data science is fun', 'big data is big', 'science uses data']
# YOUR CODE HERE

**Exercise 2 solution**

In [ ]:
practice_counts = (sc.parallelize(practice_text)
                   .flatMap(lambda line: line.split())
                   .map(lambda word: (word, 1))
                   .reduceByKey(lambda a, b: a + b))
print(sorted(practice_counts.collect()))
assert dict(practice_counts.collect())['data'] == 3

### Exercise 3. Product purchases
Given `('Pen', 2), ('Notebook', 3), ('Pen', 4), ('Pencil', 5), ('Notebook', 2)`, calculate quantities by product and sort descending by quantity, then alphabetically for ties.

In [ ]:
# Exercise 3: Write your solution here
purchases = [('Pen', 2), ('Notebook', 3), ('Pen', 4),
             ('Pencil', 5), ('Notebook', 2)]
# YOUR CODE HERE

**Exercise 3 solution**

In [ ]:
purchase_totals = (sc.parallelize(purchases)
                   .reduceByKey(lambda a, b: a + b)
                   .sortBy(lambda x: (-x[1], x[0])))
print(purchase_totals.collect())
assert purchase_totals.collect() == [('Pen', 6), ('Notebook', 5), ('Pencil', 5)]

### Exercise 4. Concept check
1. Why does `rdd[0] = 100` raise an error?
2. What causes lazy transformations to execute?
3. What role does lineage play if a partition is lost?
4. Why is a DataFrame generally preferable for SQL-like analytics?
5. Can PySpark create a strongly typed `Dataset[Person]`? Explain.

**Instructor discussion prompts:** A failed action may be retried using lineage if upstream data remains available. `.unpersist()` is not equivalent to killing an executor. An RDD is immutable even when its records contain mutable Python objects.

## Key takeaways
RDDs teach partitioned execution, transformations, actions, immutability, and lineage. `reduceByKey` supports grouped aggregation; DataFrames add schemas, SQL expressions, and Catalyst optimization. Typed Datasets are a JVM-language feature. Spark fault tolerance depends on lineage and recoverable input data, not immutability alone.

**Optional cleanup:** Run the cell below only when you are finished with the notebook.

In [ ]:
# Optional final cell: Stop Spark
# spark.stop()
# print('Spark session stopped')